# Fabric Workspace Inventory v4 — Multi-Workspace

Scans **many workspaces** in one run and writes them all into a single snapshot.

### What changed from v3.2
- `workspace_ids` read from `governance_config` (comma-separated) instead of a single parameter
- Per-workspace scan wrapped in a loop; a failure in one workspace does not stop the others
- Governance flags (duplicates, orphans) computed **per workspace** — a model in one workspace
  is not matched against a report in another
- Activity Events and Unused Artifacts are tenant-wide, so they still run once
- One `snapshot_id` covers every workspace in the run

### Permissions
| Capability | Needs |
|---|---|
| Scan any workspace | Fabric Admin (tenant-wide) |
| Delete items | Contributor/Admin **per workspace** |


## 1. Install dependencies

In [ ]:
try:
    import sempy_labs
    print("semantic-link-labs is available.")
except ImportError:
    print("Installing semantic-link-labs...")
    %pip install semantic-link-labs -q
    print("Installed. If the kernel restarted, re-run all cells from here.")


## 2. Parameters

In [ ]:
# Leave blank to read workspace_ids from governance_config.
# Set explicitly to override, e.g. "guid1,guid2"
workspace_ids_override   = ""

save_to_warehouse        = True
stale_cutoff_days        = 90
activity_lookback_days   = 30
enable_scanner_api       = True
enable_unused_artifacts  = True
enable_activity_events   = True


## 3. Setup

In [ ]:
import requests, time, uuid, logging
import pandas as pd, numpy as np
from datetime import datetime, timezone, timedelta
import notebookutils

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("inventory_v4")

FABRIC_API_BASE  = "https://api.fabric.microsoft.com/v1"
POWERBI_API_BASE = "https://api.powerbi.com/v1.0/myorg"
FABRIC_APP       = "https://app.fabric.microsoft.com"

snapshot_id       = str(uuid.uuid4())
snapshot_time_utc = datetime.now(timezone.utc).isoformat()
log.info(f"Snapshot ID: {snapshot_id}")

section_times = {}
def timed_section(name):
    class _T:
        def __enter__(self):
            self.s = time.time(); log.info(f"▶ {name}"); return self
        def __exit__(self, *a):
            e = time.time() - self.s; section_times[name] = e; log.info(f"✔ {name} ({e:.1f}s)")
    return _T()

token = notebookutils.credentials.getToken("pbi")
token_at = time.time()

def get_headers():
    global token, token_at
    if time.time() - token_at > 2400:
        log.info("Refreshing token..."); token = notebookutils.credentials.getToken("pbi"); token_at = time.time()
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def call_api(url, base="fabric", max_retries=5, caller="API"):
    for attempt in range(max_retries):
        r = requests.get(url, headers=get_headers())
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            w = int(r.headers.get("Retry-After", 5))
            log.warning(f"[{caller}] Throttled, waiting {w}s"); time.sleep(w); continue
        if r.status_code in (500, 502, 503, 504):
            # Transient server error — retry with a short fixed backoff instead of aborting
            # the whole workspace's scan on the first hiccup (there's no Retry-After on 5xx).
            log.warning(f"[{caller}] {r.status_code} (transient), retrying in 5s (attempt {attempt+1}/{max_retries})")
            time.sleep(5); continue
        if r.status_code in (401, 403):
            raise PermissionError(f"[{caller}] {r.status_code}: {r.text[:200]}")
        r.raise_for_status()
    raise RuntimeError(f"[{caller}] failed after retries")

log.info("Authenticated.")

# Warehouse connection (Governance schema in DW_Fabric) — used to read governance_config
# and persist the snapshot at the end. See Governance_Table_Setup for schema details.
import pyodbc, struct

WAREHOUSE_SQL_ENDPOINT = "<WAREHOUSE_SQL_ENDPOINT>"
WAREHOUSE_DATABASE     = "DW_Fabric"
GOVERNANCE_SCHEMA      = "Governance"

def get_warehouse_connection():
    wh_token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = wh_token.encode("utf-16-le")
    token_struct = struct.pack(f'<I{len(token_bytes)}s', len(token_bytes), token_bytes)
    SQL_COPT_SS_ACCESS_TOKEN = 1256
    conn_str = (
        f"Driver={{ODBC Driver 18 for SQL Server}};"
        f"Server={WAREHOUSE_SQL_ENDPOINT};"
        f"Database={WAREHOUSE_DATABASE};"
        f"Encrypt=Yes;TrustServerCertificate=No"
    )
    return pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})

conn = get_warehouse_connection()


## 4. Resolve workspace list

In [ ]:
with timed_section("Resolve workspaces"):
    if workspace_ids_override.strip():
        workspace_ids = [w.strip() for w in workspace_ids_override.split(",") if w.strip()]
        log.info("Using workspace_ids_override parameter.")
    else:
        try:
            cfg = pd.read_sql(f"SELECT config_key, config_value FROM {GOVERNANCE_SCHEMA}.governance_config", conn)
            cfg = dict(zip(cfg["config_key"], cfg["config_value"]))
            workspace_ids = [w.strip() for w in cfg.get("workspace_ids", "").split(",") if w.strip()]
            log.info("Read workspace_ids from governance_config.")
        except Exception as e:
            log.warning(f"Could not read governance_config: {e}")
            workspace_ids = [notebookutils.runtime.context["currentWorkspaceId"]]
            log.info("Falling back to current workspace.")

    if not workspace_ids:
        workspace_ids = [notebookutils.runtime.context["currentWorkspaceId"]]

    log.info(f"Workspaces to scan: {len(workspace_ids)}")
    for w in workspace_ids:
        log.info(f"  {w}")


## 5. Per-workspace scan

One function that does everything workspace-specific: name, Core Items, Admin Items,
Scanner API, merge, web URLs. Called once per workspace.


In [ ]:
TYPE_URL_MAP = {
    "Report":"reports","SemanticModel":"datasets","Dashboard":"dashboards","Dataflow":"dataflows",
    "DataPipeline":"pipelines","Notebook":"notebooks","Lakehouse":"lakehouses","Warehouse":"warehouses",
    "SQLEndpoint":"sqlEndpoints","Eventhouse":"eventhouses","KQLDatabase":"kqlDatabases",
    "KQLQueryset":"kqlQuerysets","KQLDashboard":"kqlDashboards","Environment":"environments",
    "SQLDatabase":"sqlDatabases","MirroredDatabase":"mirroredDatabases","Eventstream":"eventstreams",
    "Reflex":"reflexes","CopyJob":"copyJobs","SparkJobDefinition":"sparkJobDefinitions",
}

def extract_principal(p):
    if not p or not isinstance(p, dict):
        return None
    return (p.get("userDetails", {}) or {}).get("userPrincipalName") or p.get("displayName") or p.get("id")


def scan_one_workspace(ws_id):
    """Scan a single workspace. Returns a DataFrame, or raises on fatal failure."""

    # ── workspace name ─────────────────────────────────
    ws_name = ""
    try:
        ws_name = call_api(f"{FABRIC_API_BASE}/workspaces/{ws_id}", caller="Workspace").get("displayName", "")
    except Exception as e:
        log.warning(f"  [{ws_id}] name lookup failed: {e}")

    # ── Core Items ─────────────────────────────────────
    items, url = [], f"{FABRIC_API_BASE}/workspaces/{ws_id}/items"
    while url:
        d = call_api(url, caller="CoreItems")
        items.extend(d.get("value", []))
        ct = d.get("continuationToken")
        url = f"{FABRIC_API_BASE}/workspaces/{ws_id}/items?continuationToken={ct}" if ct else None

    if items:
        df_core = pd.DataFrame(items)[["id","displayName","type","description"]].rename(columns={"displayName":"name"})
    else:
        df_core = pd.DataFrame(columns=["id","name","type","description"])

    # ── Admin Items ────────────────────────────────────
    admin_raw, admin_ok = [], False
    try:
        url = f"{FABRIC_API_BASE}/admin/items?workspaceId={ws_id}"
        while url:
            d = call_api(url, caller="AdminItems")
            admin_raw.extend(d.get("itemEntities", []))
            ct = d.get("continuationToken")
            url = f"{FABRIC_API_BASE}/admin/items?workspaceId={ws_id}&continuationToken={ct}" if ct else None
        admin_ok = True
    except PermissionError:
        log.warning(f"  [{ws_name or ws_id}] no admin access — core data only")

    if admin_ok and admin_raw:
        df_admin = pd.DataFrame([{
            "id": i.get("id"),
            "created_by": extract_principal(i.get("creatorPrincipal")),
            "last_modified": i.get("lastUpdatedDate"),
            "state": i.get("state"),
            "capacity_id": i.get("capacityId"),
        } for i in admin_raw])

        admin_only = set(df_admin["id"].dropna()) - set(df_core["id"].dropna())
        name_map = {i["id"]: {"name": i.get("name", i.get("displayName","")),
                              "type": i.get("type","Unknown"),
                              "description": i.get("description","")}
                    for i in admin_raw if i.get("id") in admin_only}

        df = df_core.merge(df_admin, on="id", how="outer")
        for idx, row in df[df["name"].isna()].iterrows():
            for c, v in name_map.get(row["id"], {}).items():
                if v:
                    df.at[idx, c] = v
    else:
        df = df_core.copy()
        for c in ["created_by","last_modified","state","capacity_id"]:
            df[c] = None

    # ── Scanner API: created_date + modified_by ────────
    created_map, modby_map = {}, {}
    if enable_scanner_api and admin_ok:
        try:
            import sempy_labs.admin as sll
            res = None
            try:
                res = sll.scan_workspaces(workspace=ws_id)
            except TypeError:
                # This fallback scans the WHOLE TENANT instead of just this workspace — silently
                # doing this once per workspace in multi-workspace mode would multiply scan cost
                # by N. Log it loudly so it's never invisible if it starts triggering.
                log.error(f"  [{ws_name or ws_id}] scan_workspaces(workspace=...) not supported by "
                          f"installed semantic-link-labs — falling back to a FULL TENANT scan for "
                          f"this one workspace. This is expensive in multi-workspace mode; check the "
                          f"library version.")
                res = sll.scan_workspaces()

            if isinstance(res, dict):
                wss = res.get("workspaces", [res])
                target = next((w for w in wss if isinstance(w, dict) and str(w.get("id","")) == ws_id), None)
                if target is None and len(wss) == 1:
                    target = wss[0]
                if target:
                    for _, lst in target.items():
                        if isinstance(lst, list) and lst and isinstance(lst[0], dict):
                            for it in lst:
                                iid = str(it.get("id",""))
                                if not iid:
                                    continue
                                for k in ("createdDate","createdDateTime","CreatedDate"):
                                    if it.get(k):
                                        created_map[iid] = str(it[k]); break
                                for k in ("modifiedBy","configuredBy","createdBy"):
                                    if it.get(k):
                                        modby_map[iid] = str(it[k]); break
        except ImportError:
            pass
        except Exception as e:
            log.warning(f"  [{ws_name or ws_id}] scanner failed: {e}")

    df["created_date"] = df["id"].map(lambda x: created_map.get(str(x)) if pd.notna(x) else None)
    df["modified_by"]  = df["id"].map(lambda x: modby_map.get(str(x)) if pd.notna(x) else None)

    # ── context + urls ────────────────────────────────
    df["workspace_id"]   = ws_id
    df["workspace_name"] = ws_name
    df["web_url"] = df.apply(
        lambda r: f"{FABRIC_APP}/groups/{ws_id}/{TYPE_URL_MAP[r['type']]}/{r['id']}"
        if r.get("type") in TYPE_URL_MAP and r.get("id") else None, axis=1)

    log.info(f"  [{ws_name or ws_id}] {len(df)} items"
             f" | created_date {df['created_date'].notna().sum()}"
             f" | modified_by {df['modified_by'].notna().sum()}")
    return df


## 6. Scan every workspace

In [ ]:
frames, failed = [], []

with timed_section("Scan all workspaces"):
    for i, ws in enumerate(workspace_ids, 1):
        log.info(f"[{i}/{len(workspace_ids)}] {ws}")
        try:
            frames.append(scan_one_workspace(ws))
        except Exception as e:
            log.error(f"  FAILED: {e}")
            failed.append({"workspace_id": ws, "error": str(e)[:200]})

    if not frames:
        raise RuntimeError("No workspace could be scanned. Check permissions.")

    df_final = pd.concat(frames, ignore_index=True)
    df_final["snapshot_id"]       = snapshot_id
    df_final["snapshot_time_utc"] = snapshot_time_utc
    df_final = df_final.sort_values(["workspace_name","type","name"]).reset_index(drop=True)

    log.info(f"Scanned OK: {len(frames)}/{len(workspace_ids)} workspaces, {len(df_final)} items total")
    if failed:
        log.warning(f"Failed workspaces: {len(failed)}")
        for f in failed:
            log.warning(f"  {f['workspace_id']}: {f['error']}")


## 7. Summary by workspace

In [ ]:
summary = (df_final.groupby(["workspace_name","workspace_id"])
            .agg(items=("id","count"), types=("type","nunique"))
            .reset_index().sort_values("items", ascending=False))
print(f"\n{len(df_final)} items across {len(summary)} workspaces\n")
display(summary)

print("\nBy type across all workspaces:\n")
display(df_final.groupby("type").size().reset_index(name="count").sort_values("count", ascending=False))


## 8. Unused artifacts (tenant-wide)

`list_unused_artifacts()` covers the whole tenant, so it runs once rather than per workspace.


In [ ]:
unused_ids, unused_created, unused_accessed = set(), {}, {}

with timed_section("Unused artifacts"):
    if not enable_unused_artifacts:
        log.info("Disabled.")
    else:
        try:
            import sempy_labs.admin as sll
            du = sll.list_unused_artifacts()
            if du is None or (hasattr(du,"empty") and du.empty):
                log.info("Returned 0 items.")
            else:
                log.info(f"Returned {len(du)} items. Columns: {list(du.columns)}")
                idc = next((c for c in ["Artifact Id","Id","id","artifactId"] if c in du.columns), None)
                if idc:
                    unused_ids = set(du[idc].dropna().astype(str))
                    log.info(f"  {len(unused_ids)} unused IDs matched")

                    def to_iso(series):
                        for attempt in (lambda s: pd.to_datetime(s, unit="ms", utc=True, errors="coerce"),
                                        lambda s: pd.to_datetime(pd.to_numeric(s, errors="coerce"), unit="ms", utc=True, errors="coerce"),
                                        lambda s: pd.to_datetime(s, utc=True, errors="coerce")):
                            try:
                                r = attempt(series)
                                if r.notna().any():
                                    return r
                            except Exception:
                                pass
                        return pd.Series([pd.NaT]*len(series))

                    cc = next((c for c in ["Created Date Time","Created Date","createdDateTime"] if c in du.columns), None)
                    ac = next((c for c in ["Last Accessed Date Time","Last Accessed","lastAccessedDateTime"] if c in du.columns), None)
                    if cc:
                        for a, d in zip(du[idc].astype(str), to_iso(du[cc])):
                            if pd.notna(d): unused_created[str(a)] = d.isoformat()
                        log.info(f"  created_date for {len(unused_created)}")
                    if ac:
                        for a, d in zip(du[idc].astype(str), to_iso(du[ac])):
                            if pd.notna(d): unused_accessed[str(a)] = d.isoformat()
                        log.info(f"  last_accessed for {len(unused_accessed)}")
        except ImportError:
            log.info("semantic-link-labs not installed.")
        except Exception as e:
            log.warning(f"Failed: {e}")


## 9. Activity Events (tenant-wide)

The Power BI Activity Events API returns events for the whole tenant, so one pass covers
every workspace. Expect the oldest 2–3 days to fail with HTTP 400 — that is the retention
boundary, not an error.


In [ ]:
activity = {}

with timed_section("Activity Events"):
    if not enable_activity_events:
        log.info("Disabled.")
    else:
        try:
            end_dt   = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
            start_dt = end_dt - timedelta(days=min(activity_lookback_days, 30))
            log.info(f"{start_dt.date()} → {end_dt.date()}")

            events, day, scanned, failed_days = [], start_dt, 0, 0
            while day < end_dt:
                ds = day.strftime("%Y-%m-%d")
                url = (f"{POWERBI_API_BASE}/admin/activityevents"
                       f"?startDateTime='{ds}T00:00:00.000Z'&endDateTime='{ds}T23:59:59.000Z'")
                while url:
                    try:
                        d = call_api(url, caller="ActivityEvents")
                    except PermissionError:
                        raise
                    except Exception:
                        failed_days += 1
                        break
                    events.extend(d.get("activityEventEntities", []))
                    url = d.get("continuationUri")
                day += timedelta(days=1); scanned += 1
                if scanned % 10 == 0:
                    log.info(f"  {scanned} days, {len(events)} events...")

            log.info(f"{len(events)} events over {scanned} days ({failed_days} skipped)")

            if events:
                de = pd.DataFrame(events)
                idc = next((c for c in ["ArtifactId","artifactId","ItemId","itemId"] if c in de.columns), None)
                tc  = next((c for c in ["CreationTime","creationTime","Timestamp"] if c in de.columns), None)
                uc  = next((c for c in ["UserId","userId","UserKey"] if c in de.columns), None)
                if idc and tc:
                    de["_id"] = de[idc].astype(str)
                    de["_t"]  = pd.to_datetime(de[tc], errors="coerce", utc=True)
                    for iid, g in de.groupby("_id"):
                        lu = g["_t"].max()
                        activity[iid] = {"last_used": lu.isoformat() if pd.notna(lu) else None,
                                         "access_count": len(g),
                                         "unique_users": g[uc].nunique() if uc else None}
                    log.info(f"Usage data for {len(activity)} distinct items")
        except PermissionError:
            log.warning("Requires Fabric Admin. Skipping.")
        except Exception as e:
            log.warning(f"Failed: {e}")


## 10. Merge usage data

In [ ]:
with timed_section("Merge usage"):
    df_final["last_used_date"]   = df_final["id"].map(lambda x: activity.get(str(x),{}).get("last_used") if pd.notna(x) else None)
    df_final["access_count_30d"] = df_final["id"].map(lambda x: activity.get(str(x),{}).get("access_count") if pd.notna(x) else None)
    df_final["unique_users_30d"] = df_final["id"].map(lambda x: activity.get(str(x),{}).get("unique_users") if pd.notna(x) else None)

    df_final["is_unused_artifact"] = df_final["id"].apply(
        lambda x: 1 if (pd.notna(x) and str(x) in unused_ids) else 0)

    if unused_created:
        df_final["created_date"] = df_final.apply(
            lambda r: r["created_date"] if pd.notna(r.get("created_date"))
            else unused_created.get(str(r["id"])) if pd.notna(r.get("id")) else None, axis=1)
    if unused_accessed:
        df_final["last_used_date"] = df_final.apply(
            lambda r: r["last_used_date"] if pd.notna(r.get("last_used_date"))
            else unused_accessed.get(str(r["id"])) if pd.notna(r.get("id")) else None, axis=1)

    now = pd.Timestamp.now(tz="UTC")
    df_final["days_since_last_used"] = (now - pd.to_datetime(df_final["last_used_date"], errors="coerce", utc=True)).dt.days
    df_final["days_since_modified"]  = (now - pd.to_datetime(df_final["last_modified"],  errors="coerce", utc=True)).dt.days

    log.info(f"With activity events: {df_final['access_count_30d'].notna().sum()}")
    log.info(f"With last_used (any):  {df_final['last_used_date'].notna().sum()}")
    log.info(f"Unused artifacts:      {(df_final['is_unused_artifact']==1).sum()}")


## 11. Governance analysis — scoped per workspace

Duplicate and orphan detection run **within each workspace**. A semantic model in one
workspace must not be matched against a report in another.


In [ ]:
with timed_section("Governance analysis"):
    dsm = df_final["days_since_modified"].fillna(99999)
    dsu = df_final["days_since_last_used"].fillna(99999)
    has_usage = df_final["last_used_date"].notna()

    df_final["is_stale"] = pd.array(
        np.where(has_usage, (dsm > stale_cutoff_days) & (dsu > stale_cutoff_days), dsm > stale_cutoff_days),
        dtype="int64")

    cb = df_final["created_by"].fillna("").astype(str).str.strip().str.lower()
    df_final["has_missing_owner"] = np.where((cb=="")|(cb=="nan")|(cb=="none"), 1, 0)

    # ── duplicates: keyed per workspace ────────────────
    df_final["is_duplicate_name"] = 0
    named = df_final["name"].notna() & (df_final["name"].astype(str).str.strip() != "")
    if named.any():
        sub = df_final.loc[named].copy()
        sub["_k"] = (sub["workspace_id"].astype(str) + "||"
                     + sub["name"].str.lower().str.strip() + "||"
                     + sub["type"].str.lower().str.strip())
        dups = set(sub["_k"].value_counts().pipe(lambda s: s[s>1]).index)
        df_final.loc[named, "is_duplicate_name"] = np.where(sub["_k"].isin(dups), 1, 0)

    # ── orphans: computed per workspace ────────────────
    df_final["is_orphaned_model"]    = 0
    df_final["is_orphaned_endpoint"] = 0

    for ws, grp in df_final.groupby("workspace_id"):
        reports = set(grp.loc[grp["type"]=="Report","name"].dropna().str.lower().str.strip())
        parents = set(grp.loc[grp["type"].isin(["Lakehouse","Warehouse"]),"name"].dropna().str.lower().str.strip())

        # A SemanticModel's report is commonly named differently from the model itself,
        # so a bare name mismatch is a weak signal on its own — it can false-flag actively-used
        # models. Requiring is_unused_artifact==1 too corroborates it with Power BI's own usage
        # telemetry (a signal this project already trusts) before treating it as orphaned.
        m = (df_final["workspace_id"]==ws) & (df_final["type"]=="SemanticModel")
        if m.any():
            no_matching_report = ~df_final.loc[m,"name"].str.lower().str.strip().isin(reports)
            confirmed_unused = df_final.loc[m,"is_unused_artifact"] == 1
            df_final.loc[m,"is_orphaned_model"] = np.where(no_matching_report & confirmed_unused, 1, 0)

        e = (df_final["workspace_id"]==ws) & (df_final["type"]=="SQLEndpoint")
        if e.any():
            df_final.loc[e,"is_orphaned_endpoint"] = np.where(
                ~df_final.loc[e,"name"].str.lower().str.strip().isin(parents), 1, 0)

    # ── score ──────────────────────────────────────────
    df_final["cleanup_candidate_score"] = 0
    df_final.loc[df_final["is_stale"]==1,             "cleanup_candidate_score"] += 30
    df_final.loc[df_final["is_unused_artifact"]==1,   "cleanup_candidate_score"] += 25
    df_final.loc[df_final["has_missing_owner"]==1,    "cleanup_candidate_score"] += 15
    df_final.loc[df_final["is_duplicate_name"]==1,    "cleanup_candidate_score"] += 10
    df_final.loc[(df_final["is_orphaned_model"]==1)|(df_final["is_orphaned_endpoint"]==1),
                                                       "cleanup_candidate_score"] += 10
    df_final.loc[dsm > 180,                            "cleanup_candidate_score"] += 10

    log.info(f"Stale: {(df_final['is_stale']==1).sum()} | Not stale: {(df_final['is_stale']==0).sum()}")
    log.info(f"Missing owner: {(df_final['has_missing_owner']==1).sum()}")
    log.info(f"Duplicates: {(df_final['is_duplicate_name']==1).sum()}")
    log.info(f"Orphaned models: {(df_final['is_orphaned_model']==1).sum()}")
    log.info(f"Orphaned endpoints: {(df_final['is_orphaned_endpoint']==1).sum()}")
    log.info(f"Score >=50: {(df_final['cleanup_candidate_score']>=50).sum()}"
             f" | 30-49: {((df_final['cleanup_candidate_score']>=30)&(df_final['cleanup_candidate_score']<50)).sum()}")


## 12. Health by workspace

In [ ]:
health = (df_final.groupby("workspace_name").agg(
    items=("id","count"),
    stale=("is_stale","sum"),
    unused=("is_unused_artifact","sum"),
    no_owner=("has_missing_owner","sum"),
    high_risk=("cleanup_candidate_score", lambda s: int((s>=50).sum())),
    avg_score=("cleanup_candidate_score","mean"),
).reset_index())
health["stale_pct"] = (health["stale"]*100/health["items"]).round(1)
health["avg_score"] = health["avg_score"].round(1)
display(health.sort_values("stale_pct", ascending=False))


## 13. Persist snapshot

In [ ]:
with timed_section("Warehouse persistence"):
    if not save_to_warehouse:
        log.info("Skipped.")
    else:
        cols = ["id","name","type","description","web_url",
                "workspace_id","workspace_name","capacity_id",
                "created_by","modified_by","created_date","last_modified",
                "last_used_date","access_count_30d","unique_users_30d","is_unused_artifact",
                "days_since_modified","days_since_last_used","state",
                "is_stale","has_missing_owner","is_duplicate_name",
                "is_orphaned_model","is_orphaned_endpoint","cleanup_candidate_score",
                "snapshot_id","snapshot_time_utc"]
        cols = [c for c in cols if c in df_final.columns]
        out = df_final[cols].copy()
        try:
            wh_cursor = conn.cursor()
            wh_cursor.fast_executemany = True
            insert_sql = (
                f"INSERT INTO {GOVERNANCE_SCHEMA}.workspace_inventory_snapshot "
                f"({','.join('[' + c + ']' for c in cols)}) VALUES ({','.join(['?']*len(cols))})"
            )
            # Force plain VARCHAR(MAX) binding for every column — long description/web_url/name
            # values otherwise get auto-detected by pyodbc as a legacy LOB type, which Fabric
            # Warehouse's UTF-8 collation rejects.
            wh_cursor.setinputsizes([(pyodbc.SQL_VARCHAR, 0, 0)] * len(cols))
            wh_cursor.executemany(insert_sql, out.astype(str).values.tolist())
            conn.commit()
            log.info(f"✔ {len(out)} rows -> {GOVERNANCE_SCHEMA}.workspace_inventory_snapshot (snapshot {snapshot_id})")
        except Exception as e:
            log.error(f"Save failed: {e}")


## 14. Execution summary

In [ ]:
print("="*70)
print("  FABRIC WORKSPACE INVENTORY v4 (MULTI-WORKSPACE) — SUMMARY")
print("="*70)
print(f"  Snapshot ID:       {snapshot_id}")
print(f"  Snapshot time:     {snapshot_time_utc}")
print(f"  Workspaces OK:     {len(frames)}/{len(workspace_ids)}")
if failed:
    for f in failed:
        print(f"    FAILED {f['workspace_id']}: {f['error'][:60]}")
print(f"  Total items:       {len(df_final)}")
print(f"  Item types:        {df_final['type'].nunique()}")
print()
print("  METADATA COVERAGE")
print("  " + "-"*17)
for c in ["id","name","created_by","modified_by","created_date","last_modified","last_used_date","web_url"]:
    if c in df_final.columns:
        n = df_final[c].notna().sum()
        print(f"  {c:22s} {n:5d}/{len(df_final)} ({n/len(df_final)*100:5.1f}%)")
print()
print("  GOVERNANCE")
print("  " + "-"*10)
print(f"  Stale:              {(df_final['is_stale']==1).sum()}")
print(f"  Not stale:          {(df_final['is_stale']==0).sum()}")
print(f"  Unused artifacts:   {(df_final['is_unused_artifact']==1).sum()}")
print(f"  Missing owner:      {(df_final['has_missing_owner']==1).sum()}")
print(f"  Duplicates:         {(df_final['is_duplicate_name']==1).sum()}")
print(f"  Orphaned models:    {(df_final['is_orphaned_model']==1).sum()}")
print(f"  Orphaned endpoints: {(df_final['is_orphaned_endpoint']==1).sum()}")
print(f"  Score >=50:         {(df_final['cleanup_candidate_score']>=50).sum()}")
print()
print("  TIMING")
print("  " + "-"*6)
tot = 0
for s,e in section_times.items():
    tot += e; print(f"  {s:35s} {e:7.1f}s")
print(f"  {'-'*44}")
print(f"  {'Total':35s} {tot:7.1f}s")
print("="*70)
